# vLLM PD 分离架构指标实时曲线

解析 vLLM PD（Prefill-Decode）分离部署的日志文件，以交互式曲线图展示各节点的关键指标随时间的变化趋势。

**7 个指标：**
| 指标 | 说明 |
|---|---|
| Prefix Cache Hit Rate | 前缀缓存命中率 (%) |
| External Prefix Cache Hit Rate | 外部前缀缓存命中率 (%) |
| GPU KV Cache Usage | GPU KV 缓存使用率 (%) |
| Avg Prompt Throughput | 平均提示词吞吐量 (tokens/s) |
| Avg Gen Throughput | 平均生成吞吐量 (tokens/s) |
| Running Requests | 正在处理的请求数 |
| Waiting Requests | 等待队列中的请求数 |

> **使用方法：** 按顺序运行各单元格；图表支持缩放、悬停、隐藏/显示曲线。

In [4]:
import os

# ═══════════════════════════════════════════════
#  配置区：按需修改以下参数
# ═══════════════════════════════════════════════
TASK_NAME  = "batch_256_2"
LOG_DIR    = None   # None → 自动推断为 <workspace>/logs/<TASK_NAME>
OUTPUT_DIR = None   # None → 自动推断为 <workspace>/analysis/<TASK_NAME>
# ═══════════════════════════════════════════════

# 推断 workspace 路径（notebook 放在 <workspace>/jupyter/）
_here = os.path.abspath(os.getcwd())
# 若当前目录是 jupyter/，workspace 在上一层
_ws   = os.path.dirname(_here) if os.path.basename(_here) == "jupyter" else _here

LOG_DIR    = os.path.abspath(LOG_DIR    or os.path.join(_ws, "logs",     TASK_NAME))
OUTPUT_DIR = os.path.abspath(OUTPUT_DIR or os.path.join(_ws, "analysis", TASK_NAME))

print(f"TASK_NAME  : {TASK_NAME}")
print(f"LOG_DIR    : {LOG_DIR}  (exists={os.path.exists(LOG_DIR)})")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

TASK_NAME  : batch_256_2
LOG_DIR    : /Users/pxy/WorkSpace/logs/batch_256_2  (exists=True)
OUTPUT_DIR : /Users/pxy/WorkSpace/analysis/batch_256_2


In [5]:
%matplotlib widget

import os, re, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display

print("OK")

OK


In [6]:
# ── 指标定义（新增 External Prefix Cache Hit Rate）─────────────────────────
METRICS = [
    ("hit_rate",              "Prefix Cache Hit Rate (%)",          (0, 100)),
    ("ext_hit_rate",          "External Prefix Cache Hit Rate (%)", (0, 100)),
    ("kv_cache",              "GPU KV Cache Usage (%)",             (0, 100)),
    ("prompt_throughput",     "Avg Prompt Throughput (tok/s)",       None),
    ("gen_throughput",        "Avg Gen Throughput (tok/s)",          None),
    ("running",               "Running Requests",                    None),
    ("waiting",               "Waiting Requests",                    None),
]

COLORS = [
    "#C0392B", "#1A5276", "#1E8449", "#6C3483",
    "#D35400", "#784212", "#0E6655", "#2C3E50",
    "#8B0000", "#00008B", "#4B0082", "#B8860B",
    "#006400", "#8B4513", "#008B8B", "#36454F",
]

# ANSI 转义码清理
_ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')

# ── 解析函数（适配 PD 分离架构日志）────────────────────────────────────────
def _parse_logs():
    """解析 prefill.log / decode_*.log，返回 {role_name: DataFrame} 和 role 列表。"""
    # 发现日志文件：prefill.log, decode_0.log, decode_1.log, ...
    log_files = sorted(
        f for f in os.listdir(LOG_DIR)
        if (f.startswith("prefill") or f.startswith("decode")) and f.endswith(".log")
    )
    if not log_files:
        raise FileNotFoundError(f"No log files (prefill*.log / decode*.log) in: {LOG_DIR}")

    _data = {}
    for fname in log_files:
        # 提取角色名：prefill / decode_0 / decode_1 ...
        role = fname.replace(".log", "")
        entries = []
        with open(os.path.join(LOG_DIR, fname), encoding='utf-8', errors='replace') as f:
            for line in f:
                if "Prefix cache hit rate:" not in line:
                    continue
                # 清理 ANSI 转义码
                clean = _ANSI_RE.sub('', line)
                # 时间戳格式：MM-DD HH:MM:SS（无年份）
                ts_m      = re.search(r'(\d{2}-\d{2} \d{2}:\d{2}:\d{2})', clean)
                rate_m    = re.search(r'(?<!External )Prefix cache hit rate: ([0-9.]+)%', clean)
                ext_m     = re.search(r'External prefix cache hit rate: ([0-9.]+)%', clean)
                gen_m     = re.search(r'Avg generation throughput: ([0-9.]+) tokens/s', clean)
                kv_m      = re.search(r'GPU KV cache usage: ([0-9.]+)%', clean)
                prompt_m  = re.search(r'Avg prompt throughput: ([0-9.]+) tokens/s', clean)
                running_m = re.search(r'Running: ([0-9]+) reqs', clean)
                waiting_m = re.search(r'Waiting: ([0-9]+) reqs', clean)
                if ts_m and rate_m:
                    # 补全年份（假设当年）
                    ts_str = f"2026-{ts_m.group(1)}"
                    entries.append({
                        "timestamp":         ts_str,
                        "hit_rate":          float(rate_m.group(1)),
                        "ext_hit_rate":      float(ext_m.group(1))     if ext_m      else None,
                        "gen_throughput":    float(gen_m.group(1))     if gen_m      else None,
                        "kv_cache":          float(kv_m.group(1))      if kv_m       else None,
                        "prompt_throughput": float(prompt_m.group(1))  if prompt_m   else None,
                        "running":           int(running_m.group(1))   if running_m  else None,
                        "waiting":           int(waiting_m.group(1))   if waiting_m  else None,
                    })
        _data[role] = pd.DataFrame(entries)
    return _data, sorted(_data.keys())

# ── 创建 figure（4 行 2 列，7 个子图）──────────────────────────────────────
plt.ioff()
fig, axes = plt.subplots(4, 2, figsize=(14, 13))
fig.subplots_adjust(hspace=0.40, wspace=0.25)
fig.suptitle(f"vLLM PD Metrics Dashboard -- {TASK_NAME}", fontsize=14, fontweight="bold")

_ax_list = []
for idx, (col, title, y_range) in enumerate(METRICS):
    ax = axes[idx // 2][idx % 2]
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (s)", fontsize=8)
    ax.grid(True, alpha=0.3)
    if y_range:
        ax.set_ylim(y_range)
    _ax_list.append((ax, col, y_range))

# 隐藏第 8 个空子图
axes[3][1].set_visible(False)

_lines = {}          # {(role, metric_idx): Line2D}
_visible = {}        # {role: bool}
_role_buttons = {}   # {role: ToggleButton}
fig.tight_layout(rect=[0, 0, 1, 0.94])

# ── 图例按钮容器 ─────────────────────────────────────────────────────────────
_legend_box = widgets.HBox([], layout=widgets.Layout(flex_flow="row wrap"))

def _make_role_button(role, role_idx):
    """为某个角色创建一个 ToggleButton。"""
    btn = widgets.ToggleButton(
        value=True,
        description=role,
        layout=widgets.Layout(width="100px", height="28px"),
        style={"font_weight": "bold"},
    )
    def _on_toggle(change, _role=role):
        vis = change["new"]
        _visible[_role] = vis
        for metric_idx in range(len(METRICS)):
            key = (_role, metric_idx)
            if key in _lines:
                _lines[key].set_visible(vis)
        for metric_idx, (ax, _, _) in enumerate(_ax_list):
            visible_lines = [l for l in ax.get_lines() if l.get_visible()]
            if visible_lines:
                ax.legend(
                    handles=visible_lines,
                    labels=[l.get_label() for l in visible_lines],
                    fontsize=7, loc="upper right",
                )
            elif ax.get_legend() is not None:
                ax.get_legend().remove()
        fig.canvas.draw_idle()

    btn.observe(_on_toggle, names="value")
    return btn

def _sync_legend_buttons(roles):
    """确保图例按钮与当前角色同步。"""
    changed = False
    for role_idx, role in enumerate(roles):
        if role not in _role_buttons:
            _visible[role] = True
            btn = _make_role_button(role, role_idx)
            _role_buttons[role] = btn
            changed = True
    stale = [r for r in _role_buttons if r not in set(roles)]
    for r in stale:
        del _role_buttons[r]
        _visible.pop(r, None)
        changed = True
    if changed:
        _legend_box.children = [_role_buttons[r] for r in sorted(_role_buttons)]

# ── 刷新函数 ──────────────────────────────────────────────────────────────────
def _refresh_frame(_frame=None):
    try:
        d, roles = _parse_logs()
        now      = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        roles_set = set(roles)

        _sync_legend_buttons(roles)

        # 移除已不存在的角色线条
        stale_keys = [k for k in _lines if k[0] not in roles_set]
        legend_dirty = set()
        for key in stale_keys:
            _lines[key].remove()
            legend_dirty.add(key[1])
            del _lines[key]

        # 更新 / 新增线条
        for metric_idx, (ax, col, y_range) in enumerate(_ax_list):
            for role_idx, role in enumerate(roles):
                df    = d[role]
                color = COLORS[role_idx % len(COLORS)]
                key   = (role, metric_idx)
                if df.empty:
                    continue
                t0 = pd.to_datetime(df["timestamp"].iloc[0])
                x  = (pd.to_datetime(df["timestamp"]) - t0).dt.total_seconds().values
                y  = df[col].values

                if key in _lines:
                    _lines[key].set_xdata(x)
                    _lines[key].set_ydata(y)
                else:
                    line, = ax.plot(x, y, color=color, linewidth=1.2,
                                    label=role)
                    line.set_visible(_visible.get(role, True))
                    _lines[key] = line
                    legend_dirty.add(metric_idx)

            ax.relim()
            ax.autoscale_view()
            if y_range:
                ax.set_ylim(y_range)

            if metric_idx in legend_dirty or (ax.get_legend() is None and ax.get_lines()):
                visible_lines = [l for l in ax.get_lines() if l.get_visible()]
                if visible_lines:
                    ax.legend(
                        handles=visible_lines,
                        labels=[l.get_label() for l in visible_lines],
                        fontsize=7, loc="upper right",
                    )
                elif ax.get_legend() is not None:
                    ax.get_legend().remove()

        fig.suptitle(
            f"vLLM PD Metrics Dashboard -- {TASK_NAME}    Last refresh: {now}",
            fontsize=14, fontweight="bold",
        )
        counts = {role: len(d[role]) for role in roles}
        _status.value = f"Running | {now} | samples: {counts}"
    except Exception as e:
        _status.value = f"Error: {e}"

# ── 控件 ─────────────────────────────────────────────────────────────────────
_btn_start = widgets.Button(description="Start", button_style="success",
                            layout=widgets.Layout(width="80px"))
_btn_stop  = widgets.Button(description="Stop",  button_style="danger",
                            layout=widgets.Layout(width="80px"))
_btn_once  = widgets.Button(description="Refresh", button_style="info",
                            layout=widgets.Layout(width="80px"))
_interval  = widgets.IntSlider(value=10, min=5, max=300, step=5,
                               description="Interval(s):",
                               style={"description_width": "70px"},
                               layout=widgets.Layout(width="300px"))
_status    = widgets.Label(value="Idle")

_anim = None

def _on_start(_):
    global _anim
    if _anim is not None:
        _anim.event_source.stop()
    interval_ms = _interval.value * 1000
    _anim = FuncAnimation(fig, _refresh_frame,
                          interval=interval_ms,
                          cache_frame_data=False)
    fig.canvas.draw_idle()
    _status.value = "Started..."

def _on_stop(_):
    global _anim
    if _anim is not None:
        _anim.event_source.stop()
        _anim = None
    _status.value = "Stopped"

def _on_once(_):
    _refresh_frame()
    fig.canvas.draw_idle()

_btn_start.on_click(_on_start)
_btn_stop.on_click(_on_stop)
_btn_once.on_click(_on_once)

_toolbar = widgets.HBox([_btn_start, _btn_stop, _btn_once, _interval, _status])

# ── 首次刷新 + 显示 ──────────────────────────────────────────────────────────
_refresh_frame()
display(widgets.VBox([_toolbar, _legend_box, fig.canvas]))

## (可选) 导出 Excel 报告

运行下方单元格可同时生成与原始脚本相同的 xlsx 文件。

In [ ]:
# ── 可选：导出 Excel（需要 openpyxl）────────────────────────────────────────
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment, PatternFill
    from openpyxl.utils import get_column_letter
except ImportError:
    print("未安装 openpyxl，跳过 Excel 导出。")
    print("可运行: pip install openpyxl")
    raise SystemExit

data, roles = _parse_logs()

os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"prefix_cache_{TASK_NAME}.xlsx")

HDR_DARK  = PatternFill("solid", start_color="1F4E79")
HDR_MID   = PatternFill("solid", start_color="2E75B6")
HDR_LIGHT = PatternFill("solid", start_color="4472C4")
WHITE_FT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FT   = Font(name="Arial", size=9)
CENTER    = Alignment(horizontal="center", vertical="center")

def hdr(ws, row, col, value, fill, width=None):
    c = ws.cell(row=row, column=col, value=value)
    c.font = WHITE_FT; c.fill = fill; c.alignment = CENTER
    if width:
        ws.column_dimensions[get_column_letter(col)].width = width

wb = Workbook()
ws_raw = wb.active
ws_raw.title = "Raw Data"

col_ptr = 1
role_col_map = {}
for role in roles:
    hdr(ws_raw, 1, col_ptr,   f"{role}_timestamp",                     HDR_DARK,  width=22)
    hdr(ws_raw, 1, col_ptr+1, f"{role}_Prefix cache hit rate",         HDR_MID,   width=26)
    hdr(ws_raw, 1, col_ptr+2, f"{role}_External hit rate",             HDR_MID,   width=26)
    hdr(ws_raw, 1, col_ptr+3, f"{role}_Avg prompt throughput",         HDR_LIGHT, width=26)
    hdr(ws_raw, 1, col_ptr+4, f"{role}_Avg generation throughput",     HDR_LIGHT, width=28)
    hdr(ws_raw, 1, col_ptr+5, f"{role}_GPU KV cache usage",            HDR_MID,   width=24)
    hdr(ws_raw, 1, col_ptr+6, f"{role}_Running",                       HDR_DARK,  width=16)
    hdr(ws_raw, 1, col_ptr+7, f"{role}_Waiting",                       HDR_DARK,  width=16)
    role_col_map[role] = tuple(range(col_ptr, col_ptr + 8))
    col_ptr += 8

for role in roles:
    df = data[role]
    tc, rc, ec, pc, gc, kc, runc, wc = role_col_map[role]
    for r_i, row in enumerate(df.itertuples(index=False), start=2):
        ws_raw.cell(row=r_i, column=tc,   value=row.timestamp).font    = BODY_FT
        ws_raw.cell(row=r_i, column=rc,   value=row.hit_rate).font     = BODY_FT
        if row.ext_hit_rate is not None:
            ws_raw.cell(row=r_i, column=ec, value=row.ext_hit_rate).font = BODY_FT
        if row.prompt_throughput is not None:
            ws_raw.cell(row=r_i, column=pc, value=row.prompt_throughput).font = BODY_FT
        if row.gen_throughput is not None:
            ws_raw.cell(row=r_i, column=gc, value=row.gen_throughput).font    = BODY_FT
        if row.kv_cache is not None:
            ws_raw.cell(row=r_i, column=kc, value=row.kv_cache).font         = BODY_FT
        if row.running is not None:
            ws_raw.cell(row=r_i, column=runc, value=row.running).font        = BODY_FT
        if row.waiting is not None:
            ws_raw.cell(row=r_i, column=wc,   value=row.waiting).font        = BODY_FT

wb.save(OUTPUT_FILE)
print(f"Excel 已保存: {OUTPUT_FILE}")